In [1]:
# ============================================================
# Prefix-Discovery Bandit + Single-Policy Baselines + Replica Ladders
# Colab-ready, self-contained
# ============================================================

import math
import random
import time
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


# ============================================================
# Utilities
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def softmax(logits: np.ndarray) -> np.ndarray:
    z = logits - np.max(logits, axis=-1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)


def categorical_sample(probs: np.ndarray) -> int:
    return int(np.random.choice(len(probs), p=probs))


def clip(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))


# ============================================================
# Environment
# ============================================================

@dataclass
class PrefixBanditEnv:
    num_arms: int = 10
    horizon: int = 4
    shortcut_arm: int = 0
    secret_code: Tuple[int, ...] = (1, 6, 3, 8)
    y_s: float = 1.0
    r: Tuple[float, ...] = (0.0, 0.5, 3.0, 15.0, 100.0)

    def __post_init__(self) -> None:
        assert len(self.secret_code) == self.horizon
        assert self.secret_code[0] != self.shortcut_arm
        assert len(self.r) == self.horizon + 1

    def prefix_match_len(self, traj: List[int]) -> int:
        m = 0
        for t in range(self.horizon):
            if traj[t] == self.secret_code[t]:
                m += 1
            else:
                break
        return m

    def reward(self, traj: List[int]) -> float:
        if traj[0] == self.shortcut_arm:
            return self.y_s
        m = self.prefix_match_len(traj)
        return self.r[m]

    # --- Vectorized batch operations ---
    def batch_reward(self, trajs: np.ndarray) -> np.ndarray:
        """trajs: (N, horizon) int array. Returns (N,) float reward array."""
        N = trajs.shape[0]
        rewards = np.zeros(N, dtype=float)
        is_shortcut = trajs[:, 0] == self.shortcut_arm
        rewards[is_shortcut] = self.y_s

        non_sc = ~is_shortcut
        if non_sc.any():
            non_sc_trajs = trajs[non_sc]
            code = np.array(self.secret_code)
            match = non_sc_trajs == code[None, :]
            # prefix match: find first False in each row
            prefix_lens = np.zeros(non_sc_trajs.shape[0], dtype=int)
            for t in range(self.horizon):
                prefix_lens += match[:, t] & (prefix_lens == t)
            r_arr = np.array(self.r)
            rewards[non_sc] = r_arr[prefix_lens]
        return rewards

    def batch_prefix_match(self, trajs: np.ndarray) -> np.ndarray:
        """trajs: (N, horizon) int array. Returns (N,) int prefix lengths."""
        code = np.array(self.secret_code)
        match = trajs == code[None, :]
        prefix_lens = np.zeros(trajs.shape[0], dtype=int)
        for t in range(self.horizon):
            prefix_lens += match[:, t] & (prefix_lens == t)
        return prefix_lens

    def batch_metrics(self, trajs: np.ndarray) -> Dict[str, float]:
        """trajs: (N, horizon) int array."""
        rewards = self.batch_reward(trajs)
        prefix = self.batch_prefix_match(trajs)
        shortcut = (trajs[:, 0] == self.shortcut_arm).astype(float)
        full = (prefix == self.horizon).astype(float)
        return {
            "avg_reward": float(rewards.mean()),
            "shortcut_rate": float(shortcut.mean()),
            "prefix_ge_1": float((prefix >= 1).mean()),
            "prefix_ge_2": float((prefix >= 2).mean()),
            "prefix_ge_3": float((prefix >= 3).mean()),
            "full_code_rate": float(full.mean()),
        }


# ============================================================
# Policy / Replica
# ============================================================

@dataclass
class PolicyReplica:
    name: str
    baseline_type: str  # 'grpo', 'kl_pg', 'entropy_pg'
    beta_kl: float = 0.1
    entropy_coef: float = 0.0
    lr: float = 0.05
    num_arms: int = 10
    horizon: int = 4
    init_scale: float = 0.0

    logits: np.ndarray = field(init=False)

    def __post_init__(self) -> None:
        self.logits = np.random.randn(self.horizon, self.num_arms) * self.init_scale

    @property
    def probs(self) -> np.ndarray:
        return softmax(self.logits)

    def sample_batch(self, n: int) -> np.ndarray:
        """Vectorized sampling. Returns (n, horizon) int array."""
        p = self.probs  # (horizon, num_arms)
        trajs = np.zeros((n, self.horizon), dtype=int)
        for t in range(self.horizon):
            # vectorized categorical sampling
            cumprobs = np.cumsum(p[t])
            u = np.random.rand(n)
            trajs[:, t] = np.searchsorted(cumprobs, u)
        return trajs

    def sample_traj(self) -> List[int]:
        p = self.probs
        return [categorical_sample(p[t]) for t in range(self.horizon)]

    def logprob_traj(self, traj, probs=None) -> float:
        if probs is None:
            probs = self.probs
        return float(sum(np.log(max(probs[t, a], 1e-12)) for t, a in enumerate(traj)))

    def logprob_prefix(self, traj, prefix_len: int, probs=None) -> float:
        if probs is None:
            probs = self.probs
        return float(sum(np.log(max(probs[t, traj[t]], 1e-12)) for t in range(prefix_len)))

    def batch_logprob(self, trajs: np.ndarray, probs=None) -> np.ndarray:
        """trajs: (N, horizon) int. Returns (N,) logprobs."""
        if probs is None:
            probs = self.probs
        lp = np.zeros(trajs.shape[0], dtype=float)
        for t in range(self.horizon):
            lp += np.log(np.maximum(probs[t, trajs[:, t]], 1e-12))
        return lp

    def importance_ratio_full(self, source: "PolicyReplica", traj) -> float:
        return math.exp(self.logprob_traj(traj) - source.logprob_traj(traj))

    def importance_ratio_prefix(self, source: "PolicyReplica", traj, prefix_len: int) -> float:
        return math.exp(self.logprob_prefix(traj, prefix_len) - source.logprob_prefix(traj, prefix_len))

    def entropy(self) -> float:
        p = self.probs
        ent = -np.sum(p * np.log(np.clip(p, 1e-12, 1.0)), axis=1)
        return float(np.mean(ent))

    def first_action_probs(self) -> np.ndarray:
        return self.probs[0].copy()


# ============================================================
# Gradient helpers
# ============================================================

def traj_score_grad(probs: np.ndarray, traj, upto: Optional[int] = None) -> np.ndarray:
    horizon, num_arms = probs.shape
    if upto is None:
        upto = horizon
    grad = np.zeros((horizon, num_arms), dtype=float)
    for t in range(upto):
        grad[t] = -probs[t].copy()
        grad[t, traj[t]] += 1.0
    return grad


def kl_grad_to_uniform(probs: np.ndarray) -> np.ndarray:
    horizon, num_arms = probs.shape
    return np.full_like(probs, 1.0 / num_arms) - probs


def entropy_grad_simple(probs: np.ndarray) -> np.ndarray:
    horizon, num_arms = probs.shape
    return np.full_like(probs, 1.0 / num_arms) - probs


# ============================================================
# On-policy updates
# ============================================================

def compute_advantages(baseline_type: str, rewards: np.ndarray) -> np.ndarray:
    return rewards - rewards.mean()


def on_policy_update(
    replica: PolicyReplica,
    trajs: np.ndarray,  # (G, horizon) int
    rewards: np.ndarray,
) -> None:
    probs = replica.probs
    adv = compute_advantages(replica.baseline_type, rewards)
    G = trajs.shape[0]

    grad = np.zeros_like(replica.logits, dtype=float)

    # Vectorized REINFORCE: for each time step, accumulate
    # sum_i adv_i * (one_hot(a_i_t) - probs_t)
    for t in range(replica.horizon):
        # one_hot contribution: scatter adv into action bins
        action_adv = np.zeros(replica.num_arms, dtype=float)
        np.add.at(action_adv, trajs[:, t], adv)
        grad[t] = action_adv / G - probs[t] * (adv.sum() / G)

    # KL regularization
    grad += replica.beta_kl * kl_grad_to_uniform(probs)

    # Entropy bonus
    if replica.baseline_type == "entropy_pg":
        grad += replica.entropy_coef * entropy_grad_simple(probs)

    replica.logits += replica.lr * grad


# ============================================================
# Routing / transfer
# ============================================================

def routing_score(
    receiver: PolicyReplica,
    source: PolicyReplica,
    traj,
    source_adv: float,
    clip_c: float,
    mode: str,
    prefix_len: Optional[int] = None,
    lambda_ref: float = 0.0,
) -> float:
    if mode == "full":
        w = min(clip_c, receiver.importance_ratio_full(source, traj))
        lp_recv = receiver.logprob_traj(traj)
    elif mode == "prefix":
        assert prefix_len is not None and prefix_len > 0
        w = min(clip_c, receiver.importance_ratio_prefix(source, traj, prefix_len))
        lp_recv = receiver.logprob_prefix(traj, prefix_len)
    else:
        raise ValueError(mode)
    return w * source_adv - lambda_ref * (-lp_recv)


def off_policy_transfer_update(
    receiver: PolicyReplica,
    source: PolicyReplica,
    traj,
    source_adv: float,
    clip_c: float,
    transfer_mode: str,
    prefix_len: Optional[int] = None,
    transfer_lr_scale: float = 1.0,
) -> None:
    probs = receiver.probs

    if transfer_mode == "full":
        w = min(clip_c, receiver.importance_ratio_full(source, traj))
        grad = w * source_adv * traj_score_grad(probs, traj, upto=len(traj))
    elif transfer_mode == "prefix":
        assert prefix_len is not None and prefix_len > 0
        w = min(clip_c, receiver.importance_ratio_prefix(source, traj, prefix_len))
        grad = w * source_adv * traj_score_grad(probs, traj, upto=prefix_len)
    else:
        raise ValueError(transfer_mode)

    receiver.logits += receiver.lr * transfer_lr_scale * grad


# ============================================================
# Configs
# ============================================================

@dataclass
class LadderConfig:
    env: PrefixBanditEnv
    replicas: List[PolicyReplica]
    group_size: int = 64
    num_iters: int = 400
    eval_batch_size: int = 2000
    clip_c: float = 2.0
    routing_margin: float = 0.0
    transfer_mode: str = "full"
    routing_mode: str = "score"
    bidirectional: bool = False
    lambda_ref: float = 0.0
    transfer_lr_scale: float = 1.0
    log_every: int = 10


# ============================================================
# Training core
# ============================================================

def select_best_candidate(
    sender: PolicyReplica,
    receiver: PolicyReplica,
    trajs: np.ndarray,    # (G, H) int
    advs: np.ndarray,     # (G,)
    env: PrefixBanditEnv,
    cfg: LadderConfig,
) -> Tuple[np.ndarray, float, int]:
    """Select best trajectory from sender's group for routing to receiver."""
    prefix_lens = env.batch_prefix_match(trajs)
    rewards = env.batch_reward(trajs)

    # Cache probs once
    recv_probs = receiver.probs
    src_probs = sender.probs

    best_score = -1e18
    best_idx = 0

    for i in range(trajs.shape[0]):
        traj_i = trajs[i]
        plen = int(prefix_lens[i])

        if cfg.routing_mode == "reward":
            score = float(rewards[i])
        else:
            # Inline score computation to avoid redundant probs calls
            adv_i = float(advs[i])
            if cfg.transfer_mode == "full":
                lp_recv = float(sum(np.log(max(recv_probs[t, traj_i[t]], 1e-12)) for t in range(env.horizon)))
                lp_src = float(sum(np.log(max(src_probs[t, traj_i[t]], 1e-12)) for t in range(env.horizon)))
                w = min(cfg.clip_c, math.exp(lp_recv - lp_src))
            elif cfg.transfer_mode == "prefix" and plen > 0:
                lp_recv = float(sum(np.log(max(recv_probs[t, traj_i[t]], 1e-12)) for t in range(plen)))
                lp_src = float(sum(np.log(max(src_probs[t, traj_i[t]], 1e-12)) for t in range(plen)))
                w = min(cfg.clip_c, math.exp(lp_recv - lp_src))
            else:
                w = 0.0
                lp_recv = 0.0
            score = w * adv_i + cfg.lambda_ref * lp_recv

        if score > best_score:
            best_score = score
            best_idx = i

    return trajs[best_idx], float(advs[best_idx]), int(prefix_lens[best_idx])


def evaluate_policy(replica: PolicyReplica, env: PrefixBanditEnv, batch_size: int) -> Dict[str, float]:
    trajs = replica.sample_batch(batch_size)
    out = env.batch_metrics(trajs)
    out["entropy"] = replica.entropy()
    out["p_shortcut_arm_t1"] = float(replica.first_action_probs()[env.shortcut_arm])
    out["p_code_arm_t1"] = float(replica.first_action_probs()[env.secret_code[0]])
    return out


def run_ladder(cfg: LadderConfig, seed: int = 0, verbose: bool = True) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    set_seed(seed)
    env = cfg.env
    replicas = cfg.replicas
    M = len(replicas)

    records: List[Dict[str, Any]] = []
    t_start = time.time()

    for it in range(cfg.num_iters):
        it_start = time.time()

        all_groups: List[np.ndarray] = []
        all_rewards: List[np.ndarray] = []
        all_advs: List[np.ndarray] = []

        # === On-policy sampling + update ===
        for k in range(M):
            trajs = replicas[k].sample_batch(cfg.group_size)
            rewards = env.batch_reward(trajs)
            advs = compute_advantages(replicas[k].baseline_type, rewards)
            on_policy_update(replicas[k], trajs, rewards)
            all_groups.append(trajs)
            all_rewards.append(rewards)
            all_advs.append(advs)

        # === Local routing ===
        for k in range(M - 1):
            left = replicas[k]
            right = replicas[k + 1]

            # Forward: left -> right
            l_traj, l_adv, l_pref = select_best_candidate(
                left, right, all_groups[k], all_advs[k], env, cfg)
            r_traj, r_adv, r_pref = select_best_candidate(
                right, right, all_groups[k + 1], all_advs[k + 1], env, cfg)

            if cfg.routing_mode == "reward":
                lscore = env.reward(l_traj.tolist())
                rscore = env.reward(r_traj.tolist())
            else:
                lscore = routing_score(right, left, l_traj, l_adv, cfg.clip_c,
                    cfg.transfer_mode, l_pref if l_pref > 0 else None, cfg.lambda_ref)
                rscore = routing_score(right, right, r_traj, r_adv, cfg.clip_c,
                    cfg.transfer_mode, r_pref if r_pref > 0 else None, cfg.lambda_ref)

            if lscore > rscore + cfg.routing_margin:
                off_policy_transfer_update(right, left, l_traj, l_adv, cfg.clip_c,
                    cfg.transfer_mode, l_pref if l_pref > 0 else None, cfg.transfer_lr_scale)

            # Backward: right -> left
            if cfg.bidirectional:
                r2l_traj, r2l_adv, r2l_pref = select_best_candidate(
                    right, left, all_groups[k + 1], all_advs[k + 1], env, cfg)
                l2l_traj, l2l_adv, l2l_pref = select_best_candidate(
                    left, left, all_groups[k], all_advs[k], env, cfg)

                if cfg.routing_mode == "reward":
                    rs2 = env.reward(r2l_traj.tolist())
                    ls2 = env.reward(l2l_traj.tolist())
                else:
                    rs2 = routing_score(left, right, r2l_traj, r2l_adv, cfg.clip_c,
                        cfg.transfer_mode, r2l_pref if r2l_pref > 0 else None, cfg.lambda_ref)
                    ls2 = routing_score(left, left, l2l_traj, l2l_adv, cfg.clip_c,
                        cfg.transfer_mode, l2l_pref if l2l_pref > 0 else None, cfg.lambda_ref)

                if rs2 > ls2 + cfg.routing_margin:
                    off_policy_transfer_update(left, right, r2l_traj, r2l_adv, cfg.clip_c,
                        cfg.transfer_mode, r2l_pref if r2l_pref > 0 else None, cfg.transfer_lr_scale)

        # === Logging ===
        do_log = (it % cfg.log_every == 0) or (it == cfg.num_iters - 1)
        if do_log:
            for k, rep in enumerate(replicas):
                ev = evaluate_policy(rep, env, cfg.eval_batch_size)
                ev.update({
                    "iter": it,
                    "replica_id": k,
                    "replica_name": rep.name,
                    "baseline_type": rep.baseline_type,
                    "beta_kl": rep.beta_kl,
                    "entropy_coef": rep.entropy_coef,
                    "transfer_mode": cfg.transfer_mode,
                    "routing_mode": cfg.routing_mode,
                    "bidirectional": cfg.bidirectional,
                    "num_replicas": M,
                    "seed": seed,
                })
                records.append(ev)

            if verbose:
                elapsed = time.time() - t_start
                it_time = time.time() - it_start
                # Print compact summary of deployed (last) replica
                dep = records[-1]  # last replica logged
                print(
                    f"[iter {it:4d}/{cfg.num_iters}] "
                    f"elapsed={elapsed:6.1f}s  "
                    f"it_time={it_time:.3f}s  "
                    f"| deployed({dep['replica_name']}): "
                    f"R={dep['avg_reward']:.3f}  "
                    f"sc={dep['shortcut_rate']:.3f}  "
                    f"p≥2={dep['prefix_ge_2']:.4f}  "
                    f"full={dep['full_code_rate']:.5f}  "
                    f"ent={dep['entropy']:.3f}"
                )

    total_time = time.time() - t_start
    if verbose:
        print(f"\nTotal training time: {total_time:.1f}s ({cfg.num_iters} iters, {M} replicas)")

    df = pd.DataFrame(records)
    return df, {"replicas": replicas, "config": cfg}


# ============================================================
# Factory helpers
# ============================================================

def make_replica(name, baseline_type, beta_kl, entropy_coef, lr,
                 num_arms, horizon, init_scale=0.0):
    return PolicyReplica(
        name=name, baseline_type=baseline_type, beta_kl=beta_kl,
        entropy_coef=entropy_coef, lr=lr, num_arms=num_arms,
        horizon=horizon, init_scale=init_scale,
    )


def make_single_baseline_config(env, baseline_type="grpo", beta_kl=0.1,
    entropy_coef=0.02, lr=0.05, num_iters=400, group_size=64,
    eval_batch_size=2000):
    replica = make_replica(
        name=f"{baseline_type}_single", baseline_type=baseline_type,
        beta_kl=beta_kl, entropy_coef=entropy_coef, lr=lr,
        num_arms=env.num_arms, horizon=env.horizon, init_scale=0.0,
    )
    return LadderConfig(
        env=env, replicas=[replica], group_size=group_size,
        num_iters=num_iters, eval_batch_size=eval_batch_size,
        clip_c=2.0, routing_margin=0.0, transfer_mode="full",
        routing_mode="score", bidirectional=False, lambda_ref=0.0,
        transfer_lr_scale=1.0, log_every=10,
    )


def make_ladder_config(env, replica_specs, num_iters=400, group_size=64,
    eval_batch_size=2000, clip_c=2.0, routing_margin=0.0,
    transfer_mode="full", routing_mode="score", bidirectional=False,
    lambda_ref=0.0, transfer_lr_scale=1.0):
    replicas = []
    for i, spec in enumerate(replica_specs):
        replicas.append(make_replica(
            name=spec.get("name", f"rep_{i}"),
            baseline_type=spec.get("baseline_type", "grpo"),
            beta_kl=spec.get("beta_kl", 0.1),
            entropy_coef=spec.get("entropy_coef", 0.0),
            lr=spec.get("lr", 0.05),
            num_arms=env.num_arms, horizon=env.horizon,
            init_scale=spec.get("init_scale", 0.0),
        ))
    return LadderConfig(
        env=env, replicas=replicas, group_size=group_size,
        num_iters=num_iters, eval_batch_size=eval_batch_size,
        clip_c=clip_c, routing_margin=routing_margin,
        transfer_mode=transfer_mode, routing_mode=routing_mode,
        bidirectional=bidirectional, lambda_ref=lambda_ref,
        transfer_lr_scale=transfer_lr_scale, log_every=10,
    )


# ============================================================
# Multi-run experiment runner
# ============================================================

def rebuild_config(cfg: LadderConfig) -> LadderConfig:
    specs = [{"name": r.name, "baseline_type": r.baseline_type,
              "beta_kl": r.beta_kl, "entropy_coef": r.entropy_coef,
              "lr": r.lr, "init_scale": r.init_scale} for r in cfg.replicas]
    return make_ladder_config(
        env=cfg.env, replica_specs=specs, num_iters=cfg.num_iters,
        group_size=cfg.group_size, eval_batch_size=cfg.eval_batch_size,
        clip_c=cfg.clip_c, routing_margin=cfg.routing_margin,
        transfer_mode=cfg.transfer_mode, routing_mode=cfg.routing_mode,
        bidirectional=cfg.bidirectional, lambda_ref=cfg.lambda_ref,
        transfer_lr_scale=cfg.transfer_lr_scale,
    )


def run_many(configs: Dict[str, LadderConfig], seeds: List[int],
             verbose: bool = True) -> pd.DataFrame:
    all_dfs = []
    total_runs = len(configs) * len(seeds)
    run_idx = 0
    t0 = time.time()

    for exp_name, cfg in configs.items():
        for seed in seeds:
            run_idx += 1
            if verbose:
                print(f"\n{'='*60}")
                print(f"  Run {run_idx}/{total_runs}: {exp_name} (seed={seed})")
                print(f"  replicas={len(cfg.replicas)}, iters={cfg.num_iters}, "
                      f"group={cfg.group_size}, mode={cfg.transfer_mode}")
                print(f"{'='*60}")

            rebuilt_cfg = rebuild_config(cfg)
            df, _ = run_ladder(rebuilt_cfg, seed=seed, verbose=verbose)
            df["experiment"] = exp_name
            all_dfs.append(df)

    elapsed = time.time() - t0
    if verbose:
        print(f"\n{'#'*60}")
        print(f"  All {total_runs} runs complete in {elapsed:.1f}s")
        print(f"{'#'*60}\n")

    return pd.concat(all_dfs, ignore_index=True)


# ============================================================
# Analysis / plots
# ============================================================

def aggregate_runs(df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "avg_reward", "shortcut_rate", "prefix_ge_1", "prefix_ge_2",
        "prefix_ge_3", "full_code_rate", "entropy",
        "p_shortcut_arm_t1", "p_code_arm_t1",
    ]
    out = (
        df.groupby(["experiment", "iter", "replica_id", "replica_name"],
                    as_index=False)[metric_cols]
        .agg(["mean", "std"])
        .reset_index()
    )
    out.columns = ["_".join(c).strip("_") for c in out.columns.values]
    return out


def get_deployed_replica_df(agg_df: pd.DataFrame) -> pd.DataFrame:
    """Extract the last (deployed) replica for each experiment at each iter."""
    rows = []
    for exp in agg_df["experiment"].unique():
        sub = agg_df[agg_df["experiment"] == exp]
        max_rep = sub["replica_id"].max()
        rows.append(sub[sub["replica_id"] == max_rep])
    return pd.concat(rows, ignore_index=True)


def plot_metric(agg_df, metric, replica_id=None, title=None, save_path=None):
    plt.figure(figsize=(8, 5))
    exps = sorted(agg_df["experiment"].unique())
    for exp in exps:
        sub = agg_df[agg_df["experiment"] == exp]
        if replica_id is not None:
            sub = sub[sub["replica_id"] == replica_id]
        if len(sub) == 0:
            continue
        x = sub["iter"]
        y = sub[f"{metric}_mean"]
        s = sub[f"{metric}_std"].fillna(0.0)
        plt.plot(x, y, label=exp)
        plt.fill_between(x, y - s, y + s, alpha=0.15)

    plt.xlabel("Iteration")
    plt.ylabel(metric)
    plt.title(title or metric)
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"  Saved: {save_path}")
    plt.close()


def plot_all_deployed(agg_df, save_dir="./"):
    dep = get_deployed_replica_df(agg_df)
    metrics = [
        ("avg_reward", "Average Reward (deployed)"),
        ("shortcut_rate", "Shortcut Rate (deployed)"),
        ("prefix_ge_2", "Prefix ≥ 2 Rate (deployed)"),
        ("full_code_rate", "Full Code Rate (deployed)"),
        ("p_code_arm_t1", "P(first action = code arm)"),
        ("entropy", "Policy Entropy (deployed)"),
    ]
    for m, t in metrics:
        plot_metric(dep, m, title=t,
                    save_path=f"{save_dir}/plot_{m}.png")


def final_summary_table(df, choose_replica="last"):
    last_iter = df["iter"].max()
    sub = df[df["iter"] == last_iter].copy()
    rows = []
    for exp, g in sub.groupby("experiment"):
        if choose_replica == "last":
            chosen = g.loc[g["replica_id"].idxmax()]
        elif choose_replica == "best_reward":
            chosen = g.loc[g["avg_reward"].idxmax()]
        else:
            raise ValueError(choose_replica)
        rows.append({
            "experiment": exp,
            "replica": chosen["replica_name"],
            "avg_reward": chosen["avg_reward"],
            "shortcut_rate": chosen["shortcut_rate"],
            "prefix_ge_2": chosen["prefix_ge_2"],
            "full_code_rate": chosen["full_code_rate"],
            "entropy": chosen["entropy"],
        })
    return pd.DataFrame(rows).sort_values("avg_reward", ascending=False).reset_index(drop=True)


def final_summary_agg(df, choose_replica="last"):
    last_iter = df["iter"].max()
    sub = df[df["iter"] == last_iter].copy()
    selected = []
    for (exp, seed), g in sub.groupby(["experiment", "seed"]):
        if choose_replica == "last":
            chosen = g.loc[g["replica_id"].idxmax()]
        else:
            chosen = g.loc[g["avg_reward"].idxmax()]
        selected.append(chosen)
    sel = pd.DataFrame(selected)
    metrics = ["avg_reward", "shortcut_rate", "prefix_ge_2", "full_code_rate", "entropy"]
    out = sel.groupby("experiment")[metrics].agg(["mean", "std"])
    out.columns = ["_".join(c) for c in out.columns]
    return out.sort_values("avg_reward_mean", ascending=False)


# ============================================================
# EXPERIMENT CONFIGS
# ============================================================

env = PrefixBanditEnv(
    num_arms=10, horizon=4, shortcut_arm=0,
    secret_code=(1, 6, 3, 8),
    y_s=1.0, r=(0.0, 0.5, 3.0, 15.0, 100.0),
)


# ---------- FAST DEBUG CONFIG (< 10 seconds total) ----------
fast_configs = {
    "grpo_single": make_single_baseline_config(
        env=env, baseline_type="grpo", beta_kl=0.08,
        lr=0.05, num_iters=100, group_size=32, eval_batch_size=500,
    ),
    "entropy_single": make_single_baseline_config(
        env=env, baseline_type="entropy_pg", beta_kl=0.08,
        entropy_coef=0.03, lr=0.05, num_iters=100, group_size=32,
        eval_batch_size=500,
    ),
    "ladder_3_full": make_ladder_config(
        env=env,
        replica_specs=[
            {"name": "hot",  "beta_kl": 0.02, "lr": 0.05},
            {"name": "mid",  "beta_kl": 0.08, "lr": 0.05},
            {"name": "cold", "beta_kl": 0.20, "lr": 0.05},
        ],
        num_iters=100, group_size=32, eval_batch_size=500,
        transfer_mode="full", routing_mode="score",
        clip_c=2.0, transfer_lr_scale=0.8,
    ),
}


# ---------- FULL CONFIG (your original experiments) ----------
full_configs = {
    "grpo_single": make_single_baseline_config(
        env=env, baseline_type="grpo", beta_kl=0.08,
        lr=0.05, num_iters=400, group_size=64,
    ),
    "kl_pg_single": make_single_baseline_config(
        env=env, baseline_type="kl_pg", beta_kl=0.08,
        lr=0.05, num_iters=400, group_size=64,
    ),
    "entropy_single": make_single_baseline_config(
        env=env, baseline_type="entropy_pg", beta_kl=0.08,
        entropy_coef=0.03, lr=0.05, num_iters=400, group_size=64,
    ),
    "ladder_3_full": make_ladder_config(
        env=env,
        replica_specs=[
            {"name": "hot",  "beta_kl": 0.02, "lr": 0.05},
            {"name": "mid",  "beta_kl": 0.08, "lr": 0.05},
            {"name": "cold", "beta_kl": 0.20, "lr": 0.05},
        ],
        num_iters=400, group_size=64, transfer_mode="full",
        routing_mode="score", clip_c=2.0, transfer_lr_scale=0.8,
    ),
    "ladder_3_prefix": make_ladder_config(
        env=env,
        replica_specs=[
            {"name": "hot",  "beta_kl": 0.02, "lr": 0.05},
            {"name": "mid",  "beta_kl": 0.08, "lr": 0.05},
            {"name": "cold", "beta_kl": 0.20, "lr": 0.05},
        ],
        num_iters=400, group_size=64, transfer_mode="prefix",
        routing_mode="score", clip_c=2.0, transfer_lr_scale=0.8,
    ),
    "ladder_5_prefix_bidir": make_ladder_config(
        env=env,
        replica_specs=[
            {"name": "r1", "beta_kl": 0.01, "lr": 0.05},
            {"name": "r2", "beta_kl": 0.03, "lr": 0.05},
            {"name": "r3", "beta_kl": 0.08, "lr": 0.05},
            {"name": "r4", "beta_kl": 0.16, "lr": 0.05},
            {"name": "r5", "beta_kl": 0.30, "lr": 0.05},
        ],
        num_iters=400, group_size=64, transfer_mode="prefix",
        routing_mode="score", bidirectional=True,
        clip_c=2.0, transfer_lr_scale=0.8,
    ),
}


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    import sys

    # mode = sys.argv[1] if len(sys.argv) > 1 else "full"
    mode = "full"
    seeds = [0, 1, 2] if mode == "full" else [0]
    configs = full_configs if mode == "full" else fast_configs

    print(f"Mode: {mode} | Seeds: {seeds} | Experiments: {list(configs.keys())}")
    print()

    results = run_many(configs, seeds=seeds, verbose=True)

    agg = aggregate_runs(results)
    plot_all_deployed(agg, save_dir="./")

    print("\n=== Final Summary (deployed replica) ===")
    print(final_summary_table(results, choose_replica="last").to_string(index=False))

    if len(seeds) > 1:
        print("\n=== Aggregated over seeds ===")
        print(final_summary_agg(results, choose_replica="last").to_string())

    # Copy plots to outputs
    import shutil, glob
    for f in glob.glob("./plot_*.png"):
        shutil.copy(f, "./sample_data/")

Mode: full | Seeds: [0, 1, 2] | Experiments: ['grpo_single', 'kl_pg_single', 'entropy_single', 'ladder_3_full', 'ladder_3_prefix', 'ladder_5_prefix_bidir']


  Run 1/18: grpo_single (seed=0)
  replicas=1, iters=400, group=64, mode=full
[iter    0/400] elapsed=   0.0s  it_time=0.002s  | deployed(grpo_single): R=0.172  sc=0.098  p≥2=0.0085  full=0.00000  ent=2.303
[iter   10/400] elapsed=   0.0s  it_time=0.000s  | deployed(grpo_single): R=0.218  sc=0.110  p≥2=0.0115  full=0.00000  ent=2.302
[iter   20/400] elapsed=   0.0s  it_time=0.001s  | deployed(grpo_single): R=0.223  sc=0.105  p≥2=0.0140  full=0.00000  ent=2.302
[iter   30/400] elapsed=   0.0s  it_time=0.000s  | deployed(grpo_single): R=0.244  sc=0.103  p≥2=0.0110  full=0.00050  ent=2.302
[iter   40/400] elapsed=   0.0s  it_time=0.000s  | deployed(grpo_single): R=0.221  sc=0.127  p≥2=0.0120  full=0.00000  ent=2.301
[iter   50/400] elapsed=   0.0s  it_time=0.000s  | deployed(grpo_single): R=0.232  sc=0.123  p≥2=0.0175  full=0.00000  

OSError: [Errno 22] Invalid argument: './sample_data/'